# Overview

In this notebook, we explore the [Lending Club loan dataset](https://www.kaggle.com/wendykan/lending-club-loan-data) and try out different machine learning models as the foundation of an automated loan approval system.

# Data Exploration

In [ ]:
import numpy as np
import pandas as pd

raw_data = pd.read_csv("loan.csv")

pd.set_option('display.max_columns', 80)
raw_data.head()

In [ ]:
# For the columns with at least one null value, how many null values are there?
criteria = raw_data.isnull().sum() > 1
raw_data[criteria.index[criteria]].isnull().sum()

### Data Cleanup

Here we drop irrelevant and extremely sparse columns. Then, we deal with missing values in the columns that remain.

In [ ]:
'''
Not sure yet about:
pymnt_plan (considered irrelevant)
purpose

Also: should we consider features that have to do with the current loan status if we're predicting the grade for new loans only?
i.e. features like last_pymnt_amnt or total_pymnt
'''

irrelevant_columns = ['id',
                      'member_id',
                      'emp_title',
                      'verification_status',
                      'issue_d',
                      'loan_status',
                      'pymnt_plan',
                      'url',
                      'desc',
                      'title',
                      'zip_code',
                      'addr_state',
                      'earliest_cr_line',
                      'initial_list_status',
                      'last_pymnt_d',
                      'last_credit_pull_d',
                      'policy_code']

# append the columns with too many null values (arbitrarily set to 50,000)
criteria = raw_data.isnull().sum() > 50000
columns_with_null = list(criteria.index[criteria])
irrelevant_columns.extend(columns_with_null)

data = raw_data.drop(irrelevant_columns, axis=1)

print(data.shape)
print(data.dtypes)
data.head()

# Benchmark

Here we train a decision tree classifier as a benchmark model and try to predict `grade`. The decision tree classifier also lets us determine which features are most important, which may let us remove less important features to reduce dimensionality.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Choose to drop rows with null values for now (consider imputing them later?)
data.dropna(axis=0, inplace=True)

y = data['grade']
X = data.drop(['int_rate', 'grade', 'sub_grade'], axis=1)

# Convert categorical data into one-hot encoding
X = pd.get_dummies(X)

# Normalize values using min-max scaling
X = (X - X.min()) / (X.max() - X.min())

# Train the benchmark
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

clf = DecisionTreeClassifier(random_state=0)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print("Accuracy: {:.2f}%".format(accuracy_score(y_test, pred) * 100))

In [ ]:
important_features = sorted(zip(X.columns, clf.feature_importances_), key=lambda x: x[1], reverse=True)
for feature in important_features:
    print("{}: {:.3f}".format(feature[0], feature[1]))